# P2 · N2 — Generation Sweep

**Paper 2 — MARQ-Bench: Machine-Authored Data Quality**

This is where models actually author rules.

**Run it in PILOT mode first.** One corpus, one model, three seeds — twelve
calls, a few cents, a few minutes. Nothing in this project has yet seen a real
model-authored rule, and the pilot is where you find out whether the IR schema
holds up before committing to 640 generations. This is the week-5 checkpoint
from the protocol.

**Resumability.** Every generation appends one line to `runs.jsonl` on Drive,
flushed immediately. If Colab dies, re-run the sweep cell — completed runs are
skipped and it continues. Interrupting is safe.

**Failures are kept.** A run that errors or refuses is retried three times, then
recorded as a null result with its reason. It is never replaced by a fresh seed;
doing so would bias the sample toward well-behaved models.

**This notebook does not parse or score anything.** It captures raw text and
runs one JSON-validity smoke test. Parsing and failure coding happen in N3, and
keeping that boundary visible is part of the anti-tautology discipline.

## 1 · Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Setup

Safe to re-run at any point — that is the fix if you hit a `NameError`.

In [ ]:
import sys, json, os, datetime
from pathlib import Path

ROOT        = Path('/content/drive/MyDrive/Paper2_RuleAuthorship')
MODULES     = ROOT / 'Notebooks'
CHECKPOINTS = ROOT / 'checkpoints'
ARTIFACTS   = ROOT / 'artifacts'
RUNS        = ROOT / 'runs'

assert ROOT.exists(),    f'project folder not found: {ROOT}'
assert MODULES.exists(), f'modules folder not found: {MODULES}'
for d in (ARTIFACTS, RUNS):
    d.mkdir(parents=True, exist_ok=True)
if str(MODULES) not in sys.path:
    sys.path.insert(0, str(MODULES))

import llmauth_prompts as P
import llmauth_generate as G
import llmauth_checkpoint as CK

ckpt = CK.Checkpoint(CHECKPOINTS)
RUN_LOG_PATH = RUNS / 'runs.jsonl'

print('prompts', P.PROMPT_VERSION, '| generate', G.GENERATE_VERSION,
      '| checkpoint', CK.CHECKPOINT_VERSION)
print('run log:', RUN_LOG_PATH)

## 3 · Load the prompts built in N1

N1 saved the complete prompt text, so this notebook never touches the corpora.
The reconstructed hashes are checked against N1's — if they differ, the prompts
have drifted and the sweep must not proceed.

In [ ]:
payloads = ckpt.step('prompt_payloads', 'json',
                     lambda: (_ for _ in ()).throw(RuntimeError('run N1 first')))[0]

bundles = {}
for key, rec in payloads.items():
    corpus, cond = key.split('|')
    bundles[(corpus, cond)] = P.PromptBundle(
        system=rec['system'], user=rec['user'],
        condition=cond, corpus_id=corpus,
        prompt_version=rec['prompt_version'],
        warning_clause=rec['warning_clause'],
        sample_seed=rec['sample_seed'],
    )
    assert bundles[(corpus, cond)].sha256 == rec['prompt_sha256'], \
        f'prompt hash mismatch for {key} — prompts have drifted since N1'

CORPORA    = sorted({c for c, _ in bundles})
CONDITIONS = ('A2', 'A3', 'A4', 'A5')

print(f'{len(bundles)} prompts loaded, all hashes match N1\n')
print(f'{"corpus":<18}' + ''.join(f'{k:>9}' for k in CONDITIONS) + '   (approx tokens)')
print('-' * 58)
for c in CORPORA:
    print(f'{c:<18}' + ''.join(f'{len(bundles[(c,k)].user)//4:>9,}' for k in CONDITIONS))

## 4 · API keys

Store keys in Colab **Secrets** (the key icon in the left sidebar), not in a
cell. Add whichever you have — the sweep runs with only open-weight models if
you add none.

Suggested secret names: `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`.

In [ ]:
try:
    from google.colab import userdata
    for name in ('ANTHROPIC_API_KEY', 'OPENAI_API_KEY'):
        try:
            os.environ[name] = userdata.get(name)
            print(f'{name:<22} loaded')
        except Exception:
            print(f'{name:<22} not set')
except ImportError:
    print('not running in Colab — set keys in the environment yourself')

## 5 · Register the model panel

Fill in the model identifiers you have decided on. Record the **exact version
string** for every commercial model and pin every open-weight model by
**commit hash, not tag** — a tag can be moved and the reproducibility claim
depends on it.

Comment out any backend you are not using. The pilot needs only one.

In [ ]:
backends = {}

# ---- commercial (fill in your chosen model identifiers) --------------------
if os.environ.get('ANTHROPIC_API_KEY'):
    try:
        %pip install -q anthropic
    except Exception:
        pass
    # backends['<anthropic-model-id>'] = G.AnthropicBackend('<anthropic-model-id>')

if os.environ.get('OPENAI_API_KEY'):
    # backends['<openai-model-id>'] = G.OpenAIBackend('<openai-model-id>')
    pass

# ---- open-weight, local on the Colab GPU ----------------------------------
# Requires a GPU runtime: Runtime > Change runtime type > T4 GPU.
# backends['qwen2.5-7b'] = G.HuggingFaceBackend(
#     'Qwen/Qwen2.5-7B-Instruct', revision='<commit-hash>')

# ---- always available: costs nothing, proves the pipeline works ------------
backends['mock-1'] = G.MockBackend('mock-1')

print('registered backends:', list(backends))
if list(backends) == ['mock-1']:
    print('\n!! Only the mock backend is registered. Uncomment a real model')
    print('   above before running anything beyond a plumbing test.')

## 6 · Pricing

Set the per-million-token prices for your models so spend is tracked as you go.
Leave a model out and its cost is simply recorded as unknown.

In [ ]:
# G.PRICING['<model-id>'] = (input_usd_per_1M, output_usd_per_1M)

for m in backends:
    p = G.PRICING.get(m)
    print(f'{m:<24} {"$%.2f in / $%.2f out per 1M" % p if p else "no pricing set"}')

## 7 · Build the sweep plan

**PILOT** — one corpus, all four conditions, three seeds. Twelve calls per
model. Do this first, read the rules, then move on.

**FULL** — every corpus, every condition, ten seeds. 160 calls per model.

In [ ]:
MODE = 'pilot'          # 'pilot' or 'full'

PILOT_CORPUS = 'bank_marketing'
PILOT_SEEDS  = (1, 2, 3)
FULL_SEEDS   = tuple(range(1, 11))
TEMPERATURE  = 0.7

corpora_to_run = [PILOT_CORPUS] if MODE == 'pilot' else CORPORA
seeds          = PILOT_SEEDS if MODE == 'pilot' else FULL_SEEDS

plans = [
    G.SweepPlan(
        key=G.RunKey(corpus, cond, model_id, seed, TEMPERATURE, variant='main'),
        prompt=bundles[(corpus, cond)],
    )
    for corpus in corpora_to_run
    for cond in CONDITIONS
    for model_id in backends
    for seed in seeds
]

est_in = sum(len(p.prompt.user) // 4 for p in plans)
print(f'MODE={MODE}   {len(plans)} runs   ~{est_in:,} input tokens')
print(f'  corpora   : {corpora_to_run}')
print(f'  conditions: {list(CONDITIONS)}')
print(f'  models    : {list(backends)}')
print(f'  seeds     : {list(seeds)}')

## 8 · Run the sweep

Safe to interrupt. Re-running this cell resumes from the run log — completed
runs are skipped, nothing is duplicated, nothing is lost.

In [ ]:
log = G.RunLog(RUN_LOG_PATH)
print(f'run log currently holds {len(log)} completed runs\n')

summary = G.run_sweep(plans, backends, log,
                      max_tokens=4096,
                      sleep_between=0.0,      # raise if you hit rate limits
                      progress_every=5)

print()
for k, v in summary.items():
    print(f'  {k:<10} {v}')

## 9 · Smoke test — did the output parse as JSON?

The only check this notebook performs. It uses `json.loads` and nothing else:
real parsing, validation, and failure coding belong to N3, and keeping that
separation visible is part of the anti-tautology discipline.

A low validity rate here means the output schema in the prompt needs work —
fix that before spending a full sweep.

In [ ]:
import re

def extract_json(text):
    if not text:
        return None
    fenced = re.findall(r'```(?:json)?\s*(.*?)```', text, re.DOTALL)
    for f in fenced:
        if f.strip():
            return f.strip()
    i = text.find('{')
    return text[i:] if i >= 0 else None

records = [json.loads(l) for l in open(RUN_LOG_PATH) if l.strip()]
stats = {}
for r in records:
    k = (r['key']['model_id'], r['key']['condition'])
    s = stats.setdefault(k, {'n': 0, 'ok': 0, 'json': 0, 'rules': 0})
    s['n'] += 1
    if not r['ok']:
        continue
    s['ok'] += 1
    try:
        payload = json.loads(extract_json(r['raw_response']))
        rules = payload.get('rules', [])
        s['json'] += 1
        s['rules'] += len(rules)
    except Exception:
        pass

print(f'{"model":<20}{"cond":<6}{"runs":>6}{"ok":>5}{"json":>6}{"avg rules":>11}')
print('-' * 54)
for (model, cond), s in sorted(stats.items()):
    avg = s['rules'] / s['json'] if s['json'] else 0
    print(f'{model:<20}{cond:<6}{s["n"]:>6}{s["ok"]:>5}{s["json"]:>6}{avg:>11.1f}')

tot = sum(s['n'] for s in stats.values())
okj = sum(s['json'] for s in stats.values())
print(f'\nJSON validity: {okj}/{tot} = {okj/tot:.1%}' if tot else 'no runs yet')

## 10 · Read the rules

**This is the cell that matters.** Read the actual rules a model wrote.

For `bank_marketing`, the thing to look for is how each condition treats
`pdays`. A rule of `pdays >= 0` discards 82% of the corpus. Does the model
write it under A2? Does it still write it under A3, having been told in plain
English that `-1` means the client was not previously contacted? Does the
census under A4 change anything?

In [ ]:
SHOW_MODEL = list(backends)[0]
SHOW_SEED  = 1

for cond in CONDITIONS:
    match = [r for r in records
             if r['key']['model_id'] == SHOW_MODEL
             and r['key']['condition'] == cond
             and r['key']['seed'] == SHOW_SEED
             and r['ok']]
    if not match:
        print(f'--- {cond}: no successful run ---\n')
        continue
    try:
        rules = json.loads(extract_json(match[0]['raw_response']))['rules']
    except Exception as e:
        print(f'--- {cond}: unparseable ({e}) ---\n')
        continue
    print(f'--- {cond}  ({len(rules)} rules) ---')
    for r in rules:
        params = json.dumps(r.get('parameters', {}))
        print(f"  {r.get('column','?'):<18} {r.get('predicate_type','?'):<14} {params}")
        if r.get('rationale'):
            print(f"      {r['rationale'][:100]}")
    print()

## 11 · Where did `pdays` land?

A focused view of the sentinel trap across conditions and seeds. Any rule with
a `min` above `-1` rejects the 82% of records that carry the sentinel.

In [ ]:
TRAPS = {
    'bank_marketing':   'pdays',
    'diabetes_130us':   'max_glu_serum',
    'nyc_tlc_yellow':   'RatecodeID',
    'online_retail_ii': 'Customer ID',
}

print(f'{"corpus":<18}{"cond":<6}{"model":<18}{"seed":>5}  rule on trap column')
print('-' * 78)
for r in records:
    if not r['ok']:
        continue
    corpus = r['key']['corpus_id']
    trap = TRAPS.get(corpus)
    try:
        rules = json.loads(extract_json(r['raw_response']))['rules']
    except Exception:
        continue
    hits = [x for x in rules if x.get('column') == trap]
    for h in hits:
        print(f'{corpus:<18}{r["key"]["condition"]:<6}{r["key"]["model_id"]:<18}'
              f'{r["key"]["seed"]:>5}  {h.get("predicate_type")} '
              f'{json.dumps(h.get("parameters", {}))}')

## 12 · Provenance and status

In [ ]:
def build_provenance():
    return {
        'notebook': 'P2_N2_generation_sweep',
        'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        'generate_version': G.GENERATE_VERSION,
        'prompt_version': P.PROMPT_VERSION,
        'mode': MODE,
        'temperature': TEMPERATURE,
        'models': list(backends),
        'pricing': {k: list(v) for k, v in G.PRICING.items()},
        'run_log': str(RUN_LOG_PATH),
        'summary': log.summary(),
    }

prov = build_provenance()
(ARTIFACTS / f'N2_provenance_{MODE}.json').write_text(json.dumps(prov, indent=2, default=str))
print(json.dumps(prov['summary'], indent=2))
print('\nwrote', ARTIFACTS / f'N2_provenance_{MODE}.json')
print('\nWhen the pilot looks right, set MODE = \'full\' in section 7 and re-run.')